# DOTA Pipeline (Colab)

This notebook mirrors the VisDrone Colab flow:
1. Mount Google Drive.
2. Copy dataset into `MyDrive` (idempotent).
3. Optionally extract ZIP archives if needed.
4. Build runtime config.
5. Run smoke pipeline.
6. Optionally run training/eval.


In [1]:
# Step 0: install dependencies in Colab runtime.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm


In [2]:
# Step 1: mount Google Drive and define paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
DRIVE_ROOT = Path('/content/drive/MyDrive')

DATASET_NAME = 'DOTA'
TARGET_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'dota_yolo'
TARGET_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)

# Source can be:
# - prepared YOLO dataset folder in Drive (recommended)
# - zip archives folder in Drive (optional unzip step below)
SOURCE_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'incoming' / 'dota_yolo'
SOURCE_ZIP_DIR = DRIVE_ROOT / 'datasets' / f'{DATASET_NAME.lower()}_zips'

# All experiment outputs will be persisted in Google Drive.
EXPERIMENTS_DRIVE_ROOT = DRIVE_ROOT / 'experiments' / 'small_objects_auto_aug' / DATASET_NAME.lower()
RUNS_DRIVE_DIR = EXPERIMENTS_DRIVE_ROOT / 'runs'
ARTIFACTS_DRIVE_DIR = EXPERIMENTS_DRIVE_ROOT / 'artifacts'
AUTO_SYNC_RESULTS_TO_DRIVE = True

RUNS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print('TARGET_DATASET_ROOT =', TARGET_DATASET_ROOT)
print('SOURCE_DATASET_ROOT =', SOURCE_DATASET_ROOT)
print('SOURCE_ZIP_DIR =', SOURCE_ZIP_DIR)
print('RUNS_DRIVE_DIR =', RUNS_DRIVE_DIR)
print('ARTIFACTS_DRIVE_DIR =', ARTIFACTS_DRIVE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TARGET_DATASET_ROOT = /content/drive/MyDrive/datasets/dota_yolo
SOURCE_DATASET_ROOT = /content/drive/MyDrive/datasets/incoming/dota_yolo
SOURCE_ZIP_DIR = /content/drive/MyDrive/datasets/dota_zips


In [3]:
from pathlib import Path
import zipfile, shutil, os

DRIVE_ROOT = Path('/content/drive/MyDrive')

TARGET_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'dota_yolo'
SOURCE_ZIP_DIR = DRIVE_ROOT / 'datasets' / 'dota_zips'

SOURCE_ZIP_DIR.mkdir(parents=True, exist_ok=True)
TARGET_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)

zip_path = SOURCE_ZIP_DIR / 'DOTAv1.zip'

# Готовый DOTA v1 в YOLO/Ultralytics формате
url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/DOTAv1.zip'

if not zip_path.exists():
    !wget -O "{zip_path}" "{url}"
else:
    print('[SKIP] ZIP already exists:', zip_path)

# Распаковка
tmp_extract = DRIVE_ROOT / 'datasets' / '_dota_extract_tmp'

if tmp_extract.exists():
    shutil.rmtree(tmp_extract)

tmp_extract.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(tmp_extract)

# Внутри архива обычно папка DOTAv1
candidates = [
    tmp_extract / 'DOTAv1',
    tmp_extract / 'dota1',
    tmp_extract,
]

src = None
for c in candidates:
    if (c / 'images').exists() and (c / 'labels').exists():
        src = c
        break

if src is None:
    raise RuntimeError('Не найдена YOLO-структура images/labels после распаковки')

if TARGET_DATASET_ROOT.exists():
    shutil.rmtree(TARGET_DATASET_ROOT)

shutil.copytree(src, TARGET_DATASET_ROOT)
shutil.rmtree(tmp_extract)

print('[OK] Dataset saved to:', TARGET_DATASET_ROOT)
print('train images:', len(list((TARGET_DATASET_ROOT / 'images' / 'train').glob('*'))))
print('val images:', len(list((TARGET_DATASET_ROOT / 'images' / 'val').glob('*'))))
print('train labels:', len(list((TARGET_DATASET_ROOT / 'labels' / 'train').glob('*'))))
print('val labels:', len(list((TARGET_DATASET_ROOT / 'labels' / 'val').glob('*'))))

[SKIP] ZIP already exists: /content/drive/MyDrive/datasets/dota_zips/DOTAv1.zip
[OK] Dataset saved to: /content/drive/MyDrive/datasets/dota_yolo
train images: 1411
val images: 458
train labels: 1411
val labels: 458


In [4]:
# Step 2: clone project repo if needed.
import subprocess
import sys

REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


Using PROJECT_ROOT: /content/small_objects_auto_aug


In [5]:
# Step 3: copy prepared dataset to MyDrive target (idempotent and safe).
import shutil

FORCE_RECOPY = False
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def _count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def _is_ready_yolo(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in req):
        return False
    return _count_images(root / 'images' / 'train') > 0 and _count_images(root / 'images' / 'val') > 0


src_resolved = SOURCE_DATASET_ROOT.resolve() if SOURCE_DATASET_ROOT.exists() else SOURCE_DATASET_ROOT
dst_resolved = TARGET_DATASET_ROOT.resolve() if TARGET_DATASET_ROOT.exists() else TARGET_DATASET_ROOT

if _is_ready_yolo(TARGET_DATASET_ROOT) and not FORCE_RECOPY:
    print('[SKIP] Target dataset already exists and looks ready.')
elif src_resolved == dst_resolved:
    # Source and target point to the same directory. Do not delete/copy over itself.
    if not _is_ready_yolo(TARGET_DATASET_ROOT):
        raise FileNotFoundError(
            f'SOURCE_DATASET_ROOT == TARGET_DATASET_ROOT but dataset is not ready: {TARGET_DATASET_ROOT}\n'
            'Either place a prepared YOLO dataset there, or run ZIP extraction step.'
        )
    print('[SKIP] Source and target are the same folder; dataset already in place.')
else:
    if not SOURCE_DATASET_ROOT.exists():
        raise FileNotFoundError(
            f'SOURCE_DATASET_ROOT not found: {SOURCE_DATASET_ROOT}\n'
            'Place prepared YOLO dataset there or use Step 4 to unpack ZIPs.'
        )

    if TARGET_DATASET_ROOT.exists():
        shutil.rmtree(TARGET_DATASET_ROOT)
    shutil.copytree(SOURCE_DATASET_ROOT, TARGET_DATASET_ROOT)
    print('[OK] Dataset copied to:', TARGET_DATASET_ROOT)

print('train images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'train'))
print('val images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'val'))


[SKIP] Target dataset already exists and looks ready.
train images: 1411
val images: 458


In [6]:
# Step 4 (optional): unpack ZIP dataset in place (if source is ZIPs).
# Use this step when SOURCE_ZIP_DIR contains archives and SOURCE_DATASET_ROOT is absent.
import zipfile
import shutil

RUN_UNZIP_FROM_SOURCE_ZIPS = False
FORCE_UNZIP = False

if RUN_UNZIP_FROM_SOURCE_ZIPS:
    if _is_ready_yolo(TARGET_DATASET_ROOT) and not FORCE_UNZIP:
        print('[SKIP] Target dataset already prepared. Set FORCE_UNZIP=True to rebuild from ZIPs.')
    else:
        if not SOURCE_ZIP_DIR.exists():
            raise FileNotFoundError(f'ZIP source dir not found: {SOURCE_ZIP_DIR}')

        zip_files = sorted(SOURCE_ZIP_DIR.glob('*.zip'))
        if not zip_files:
            raise FileNotFoundError(f'No .zip files in: {SOURCE_ZIP_DIR}')

        if TARGET_DATASET_ROOT.exists() and FORCE_UNZIP:
            shutil.rmtree(TARGET_DATASET_ROOT)
        TARGET_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

        for z in zip_files:
            print('[UNZIP]', z.name)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(TARGET_DATASET_ROOT)

        print('[OK] Unzip finished:', TARGET_DATASET_ROOT)
else:
    print('ZIP extraction skipped. Set RUN_UNZIP_FROM_SOURCE_ZIPS=True if needed.')


ZIP extraction skipped. Set RUN_UNZIP_FROM_SOURCE_ZIPS=True if needed.


In [7]:
# Step 5: build runtime config from template.
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'dota_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_runtime_config.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['kind'] = 'yolo_generic'
cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(TARGET_DATASET_ROOT)

# Persist training runs directly to Google Drive.
if isinstance(cfg.get('training'), dict):
    cfg['training']['project_dir'] = str(RUNS_DRIVE_DIR)
    cfg['training']['data_yaml'] = str(TARGET_DATASET_ROOT / 'dataset.yaml')

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('Runtime config:', runtime_cfg_path)


Runtime config: /content/small_objects_auto_aug/artifacts/dota_runtime_config.yaml


In [8]:
# Step 6: smoke run (subset -> analyze -> policy).
from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import load_yaml

cfg = load_yaml(runtime_cfg_path)

subset_root = PROJECT_ROOT / 'datasets' / f'{DATASET_NAME.lower()}_smoke'
smoke_stats_dir = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_smoke' / 'stats'
smoke_policy_dir = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_smoke' / 'policy'

subset_report = build_yolo_subset(
    dataset_root=TARGET_DATASET_ROOT,
    output_root=subset_root,
    train_images=120,
    val_images=40,
    seed=42,
    clean_output=True,
)
print('subset report:', subset_report)

stats = analyze_dataset(
    dataset_root=subset_root,
    output_dir=smoke_stats_dir,
    splits=('train',),
    config=DatasetAnalyzerConfig(
        small_max_area=float(cfg['analysis']['small_max_area']),
        medium_max_area=float(cfg['analysis']['medium_max_area']),
        tiny_max_area=float(cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
    ),
)

policy, decision_report = generate_policy_from_stats(
    stats,
    cfg=RuleEngineConfig.from_project_config(cfg),
)
paths = save_policy_artifacts(policy, decision_report, output_dir=smoke_policy_dir)

print('[OK] smoke done')
print('stats:', smoke_stats_dir / 'dataset_stats.json')
print('policy:', paths['policy_json'])


subset report: SubsetBuildReport(train_images=120, val_images=40, output_root='/content/small_objects_auto_aug/datasets/dota_smoke')


Analyze train:   0%|          | 0/120 [00:00<?, ?img/s]

[OK] smoke done
stats: /content/small_objects_auto_aug/artifacts/dota_smoke/stats/dataset_stats.json
policy: /content/small_objects_auto_aug/artifacts/dota_smoke/policy/policy_adaptive.json


In [9]:
# Step 7: full pipeline run with runtime optimization presets.
from pathlib import Path
import shutil
import yaml
from src.pipeline_mvp import run_mvp_pipeline

assert (PROJECT_ROOT / 'configs' / 'baseline.yaml').exists(), 'Missing configs/baseline.yaml'
assert (PROJECT_ROOT / 'configs' / 'manual.yaml').exists(), 'Missing configs/manual.yaml'

# Optional speed-up: mirror dataset from Drive to local Colab SSD.
USE_LOCAL_RUNTIME_DATASET = True
LOCAL_DATASET_ROOT = Path('/content/datasets/dota_yolo')

if USE_LOCAL_RUNTIME_DATASET:
    train_marker = LOCAL_DATASET_ROOT / 'images' / 'train'
    val_marker = LOCAL_DATASET_ROOT / 'images' / 'val'
    if not (train_marker.exists() and val_marker.exists()):
        print(f'[INFO] Copy dataset to local runtime: {TARGET_DATASET_ROOT} -> {LOCAL_DATASET_ROOT}')
        if LOCAL_DATASET_ROOT.exists():
            shutil.rmtree(LOCAL_DATASET_ROOT)
        LOCAL_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(TARGET_DATASET_ROOT, LOCAL_DATASET_ROOT)
    else:
        print(f'[OK] Reuse local dataset: {LOCAL_DATASET_ROOT}')

# Modes:
# quick_check: fastest sanity run
# hour_safe: usually about ~1 hour on T4/L4, no ablations
# hour_full: heavy full run (can be multiple hours)
EXECUTION_MODE = 'hour_safe'

mode_cfg = {
    'quick_check': {
        'run_training': True,
        'run_eval': False,
        'train_profile': 'fast',
        'run_ablation': False,
        'auto_predict_for_eval': False,
        'val_during_train': False,
        'cache': 'disk',
        'patience': 8,
        'mode_overrides': {
            'baseline': {'epochs': 6, 'fraction': 0.2, 'imgsz': 768, 'val': False},
            'manual': {'epochs': 6, 'fraction': 0.2, 'imgsz': 768, 'val': False},
            'adaptive': {'epochs': 8, 'fraction': 0.3, 'imgsz': 768, 'val': True},
        },
    },
    'hour_safe': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'hour',
        'run_ablation': False,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 12,
        'mode_overrides': {
            'baseline': {'epochs': 10, 'fraction': 0.5, 'imgsz': 896, 'val': False},
            'manual': {'epochs': 10, 'fraction': 0.5, 'imgsz': 896, 'val': False},
            'adaptive': {'epochs': 24, 'fraction': 1.0, 'imgsz': 960, 'val': True},
        },
    },
    'hour_full': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'hour',
        'run_ablation': True,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 20,
        'mode_overrides': {},
    },
}

assert EXECUTION_MODE in mode_cfg, f'Unknown EXECUTION_MODE: {EXECUTION_MODE}'
selected = mode_cfg[EXECUTION_MODE]

# Update runtime config before run.
with runtime_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg.setdefault('dataset', {})
cfg.setdefault('training', {})

if USE_LOCAL_RUNTIME_DATASET:
    cfg['dataset']['root'] = str(LOCAL_DATASET_ROOT)

cfg['training']['run_ablation'] = bool(selected['run_ablation'])
cfg['training']['project_dir'] = str(RUNS_DRIVE_DIR)
cfg['training']['val_during_train'] = bool(selected['val_during_train'])
cfg['training']['cache'] = selected['cache']
cfg['training']['patience'] = int(selected['patience'])
cfg['training']['mode_overrides'] = selected['mode_overrides']

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

# Rough ETA estimate.
profile_epochs = {
    'fast': int(cfg['training'].get('epochs_fast', 10)),
    'final': int(cfg['training'].get('epochs_final', 80)),
    'balanced': 15,
    'quality': 25,
    'hour': 40,
    'max_quality': 60,
}
base_epochs = profile_epochs.get(selected['train_profile'], 40)
mode_ovr = selected.get('mode_overrides', {})
train_modes = ['baseline', 'manual', 'adaptive']
if selected['run_ablation']:
    train_modes += ['adaptive_no_mosaic', 'adaptive_no_custom_albu']

def _mode_epochs(mode_name: str) -> int:
    item = mode_ovr.get(mode_name, {}) if isinstance(mode_ovr, dict) else {}
    return int(item.get('epochs', base_epochs))

def _mode_val(mode_name: str) -> bool:
    item = mode_ovr.get(mode_name, {}) if isinstance(mode_ovr, dict) else {}
    return bool(item.get('val', selected['val_during_train']))

train_epoch_minutes = 3.8
val_epoch_minutes = 0.45
post_eval_minutes = 2.0 if selected['run_eval'] else 0.0
eta_min = 0.0
for m in train_modes:
    e = _mode_epochs(m)
    eta_min += e * train_epoch_minutes
    if _mode_val(m):
        eta_min += e * val_epoch_minutes
if selected['run_eval']:
    eta_min += len(train_modes) * post_eval_minutes

print(f'[INFO] EXECUTION_MODE={EXECUTION_MODE}')
print(f'[INFO] train_profile={selected["train_profile"]}, run_ablation={selected["run_ablation"]}')
print(f'[INFO] train modes={train_modes}')
print(f'[INFO] estimated time ~ {eta_min:.0f} min (~{eta_min/60:.1f} h)')
print(f'[INFO] training.project_dir={cfg["training"]["project_dir"]}')
print(f'[INFO] dataset.root={cfg["dataset"]["root"]}')

outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=bool(selected['run_training']),
    run_eval=bool(selected['run_eval']),
    train_profile=str(selected['train_profile']),
    auto_predict_for_eval=bool(selected['auto_predict_for_eval']),
)
print(outputs)

# Persist artifacts to Drive (runs are already written to RUNS_DRIVE_DIR).
if AUTO_SYNC_RESULTS_TO_DRIVE:
    local_artifacts = PROJECT_ROOT / 'artifacts'
    if local_artifacts.exists():
        shutil.copytree(local_artifacts, ARTIFACTS_DRIVE_DIR, dirs_exist_ok=True)
    print('[OK] Artifacts synced to:', ARTIFACTS_DRIVE_DIR)

manifest_path = Path(outputs.get('experiment_manifest', ''))
if manifest_path.exists():
    manifest_copy = ARTIFACTS_DRIVE_DIR / 'experiment_manifest.json'
    shutil.copy2(manifest_path, manifest_copy)
    print('[OK] Manifest copied:', manifest_copy)


Analyze train:   0%|          | 0/1411 [00:00<?, ?img/s]

Analyze val:   0%|          | 0/458 [00:00<?, ?img/s]

FileNotFoundError: [Errno 2] No such file or directory: 'configs/baseline.yaml'